---

# 🔗 Teleportación cuántica y estados de Bell

### Entrelazamiento, Alice, Bob y comunicación clásica

**INTRODUCCIÓN A LA PROGRAMACIÓN CUÁNTICA · CENIDET**  
**TECNOLÓGICO NACIONAL DE MÉXICO**

`NIVEL INICIAL` · `QISKIT` · `GOOGLE COLAB`

---

**Oscar Alejandro López Campero**  
Agosto 2026

---

## 🧭 Ruta del módulo

Avanzaremos en dos partes:

| Parte | Tema | Meta |
|:--:|---|---|
| **A** | Estados de Bell | Construir Φ⁺, Φ⁻, Ψ⁺ y Ψ⁻ paso a paso |
| **B** | Teleportación | Seguir el estado desde Alice hasta Bob |

> No necesitamos memorizar el circuito. La meta es reconocer qué hace cada bloque.

---

# 🛠️ Preparación

Ejecuta las siguientes celdas antes de comenzar.

In [ ]:
# Aer permite simular mediciones y correcciones condicionadas.
%pip install -q "qiskit[visualization]" qiskit-aer

In [ ]:
# Herramientas que utilizaremos.
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from IPython.display import display

simulador = AerSimulator()
print("✅ Entorno preparado")

---

# 🟦 PARTE A

# 🔔 Los cuatro estados de Bell

Un estado de Bell utiliza dos qubits. Al medirlos, sus resultados están relacionados:

- **Φ:** los resultados coinciden (`00` o `11`).
- **Ψ:** los resultados son diferentes (`01` o `10`).
- Los signos **+** y **−** indican una diferencia de fase.

Comenzaremos siempre en `|00⟩`.

## 1. Estado Φ⁺

### Paso a paso

| Paso | Operación | Estado |
|:--:|---|---|
| 0 | Inicio | ∣00⟩ |
| 1 | H sobre `q₀` | (∣00⟩ + ∣01⟩) / √2 |
| 2 | CNOT `q₀ → q₁` | (∣00⟩ + ∣11⟩) / √2 |

H crea la superposición y CNOT relaciona los dos qubits.

In [ ]:
# Crear dos qubits. Los dos comienzan en |0⟩.
phi_mas = QuantumCircuit(2)

# PASO 1: H crea superposición en q0.
phi_mas.h(0)

# PASO 2: q0 controla el cambio de q1.
phi_mas.cx(0, 1)

# Mostrar el circuito y el estado final.
estado_phi_mas = Statevector.from_instruction(phi_mas)
display(phi_mas.draw('mpl'))
display(estado_phi_mas.draw('latex'))

### Comprobar Φ⁺ con mediciones

Mediremos una copia del circuito. Esperamos aproximadamente la mitad `00` y la mitad `11`.

In [ ]:
# Copiar el circuito para no modificar el original.
phi_mas_medido = phi_mas.copy()
phi_mas_medido.measure_all()

# Ejecutar 1,000 veces.
preparado_phi_mas = transpile(phi_mas_medido, simulador)
conteos_phi_mas = simulador.run(
    preparado_phi_mas,
    shots=1000,
    seed_simulator=2026
).result().get_counts()

print(conteos_phi_mas)
display(plot_histogram(conteos_phi_mas, title="Estado Φ⁺"))

---

## 2. Estado Φ⁻

Partimos de la misma receta de Φ⁺ y agregamos Z.

| Paso | Operación | Resultado |
|:--:|---|---|
| 1 | H sobre `q₀` | Crear superposición |
| 2 | CNOT `q₀ → q₁` | Crear Φ⁺ |
| 3 | Z sobre `q₀` | Cambiar el signo de ∣11⟩ |

El estado final es `(∣00⟩ − ∣11⟩) / √2`.

In [ ]:
phi_menos = QuantumCircuit(2)

# PASOS 1 y 2: crear Φ⁺.
phi_menos.h(0)
phi_menos.cx(0, 1)

# PASO 3: Z cambia la fase.
phi_menos.z(0)

estado_phi_menos = Statevector.from_instruction(phi_menos)
display(phi_menos.draw('mpl'))
display(estado_phi_menos.draw('latex'))

> 💡 Si medimos Φ⁺ y Φ⁻ directamente, ambos producen `00` y `11`. El signo menos representa una fase y no se distingue con este histograma sencillo.

---

## 3. Estado Ψ⁺

Ahora queremos resultados opuestos: `01` o `10`.

| Paso | Operación | Resultado |
|:--:|---|---|
| 1 | H sobre `q₀` | Crear superposición |
| 2 | CNOT `q₀ → q₁` | Crear Φ⁺ |
| 3 | X sobre `q₀` | Cambiar Φ⁺ por Ψ⁺ |

El estado final es `(∣01⟩ + ∣10⟩) / √2`.

In [ ]:
psi_mas = QuantumCircuit(2)

# PASOS 1 y 2: crear Φ⁺.
psi_mas.h(0)
psi_mas.cx(0, 1)

# PASO 3: X cambia las posiciones de los estados.
psi_mas.x(0)

estado_psi_mas = Statevector.from_instruction(psi_mas)
display(psi_mas.draw('mpl'))
display(estado_psi_mas.draw('latex'))

---

## 4. Estado Ψ⁻

Primero creamos Ψ⁺ y después agregamos Z para cambiar la fase.

| Paso | Operación | Resultado |
|:--:|---|---|
| 1 | H sobre `q₀` | Crear superposición |
| 2 | CNOT `q₀ → q₁` | Crear Φ⁺ |
| 3 | X sobre `q₀` | Cambiar a Ψ⁺ |
| 4 | Z sobre `q₁` | Cambiar a Ψ⁻ |

El estado final es `(∣01⟩ − ∣10⟩) / √2`.

In [ ]:
psi_menos = QuantumCircuit(2)

# PASOS 1 y 2: crear Φ⁺.
psi_menos.h(0)
psi_menos.cx(0, 1)

# PASO 3: cambiar Φ⁺ por Ψ⁺.
psi_menos.x(0)

# PASO 4: agregar la fase negativa.
psi_menos.z(1)

estado_psi_menos = Statevector.from_instruction(psi_menos)
display(psi_menos.draw('mpl'))
display(estado_psi_menos.draw('latex'))

## Comparación rápida

| Estado | Compuertas después de H y CNOT | Resultados al medir |
|:--:|---|:--:|
| Φ⁺ | Ninguna | `00`, `11` |
| Φ⁻ | Z | `00`, `11` |
| Ψ⁺ | X | `01`, `10` |
| Ψ⁻ | X y Z | `01`, `10` |

> Los signos + y − cambian la fase. Las letras Φ y Ψ cambian si las mediciones coinciden o son opuestas.

---

# 🟩 PARTE B

# 🚀 Protocolo de teleportación

La teleportación cuántica transfiere el **estado** de un qubit. No mueve físicamente el qubit y no transporta personas.

Imaginemos dos participantes:

- **Alice** quiere enviar un estado cuántico.
- **Bob** quiere recibir ese estado.

## ¿Qué tiene cada persona?

| Elemento | Propietario | Función |
|---|---|---|
| Qubit `Mensaje` | Alice | Contiene el estado que se quiere enviar |
| Qubit `Alice` | Alice | Es la mitad de Alice del estado de Bell |
| Qubit `Bob` | Bob | Es la mitad de Bob y recibirá el estado |

Antes de separarse, Alice y Bob comparten un estado de Bell:

```text
Alice: Mensaje + mitad A del par Bell
                         ║
Bob:               mitad B del par Bell
```

## La historia completa en seis pasos

| Paso | Persona | Acción |
|:--:|:--:|---|
| 1 | Alice | Prepara el estado que quiere enviar |
| 2 | Alice y Bob | Comparten un estado de Bell |
| 3 | Alice | Combina el mensaje con su mitad del par |
| 4 | Alice | Mide sus dos qubits |
| 5 | Alice | Envía dos bits clásicos a Bob |
| 6 | Bob | Aplica correcciones y recupera el estado |

Ahora construiremos cada paso en una celda distinta.

---

## Paso 0 — Crear registros con nombres

Usaremos nombres en lugar de solamente `q₀`, `q₁` y `q₂`. Así será más fácil identificar a quién pertenece cada línea.

In [ ]:
# Tres registros cuánticos, cada uno con un qubit.
mensaje = QuantumRegister(1, 'Mensaje')
qubit_alice = QuantumRegister(1, 'Alice')
qubit_bob = QuantumRegister(1, 'Bob')

# Alice guardará dos mediciones; Bob guardará una.
bits_alice = ClassicalRegister(2, 'bits_Alice')
bit_bob = ClassicalRegister(1, 'bit_Bob')

# Crear el circuito todavía vacío.
teleportacion = QuantumCircuit(
    mensaje,
    qubit_alice,
    qubit_bob,
    bits_alice,
    bit_bob
)

display(teleportacion.draw('mpl'))

## Paso 1 — Alice prepara el mensaje

Usaremos `|1⟩` porque es fácil comprobarlo. Alice aplica X al qubit `Mensaje`.

Al terminar la teleportación, Bob deberá obtener `1`.

In [ ]:
# Preparar el mensaje |1⟩.
teleportacion.x(mensaje[0])
teleportacion.barrier()

display(teleportacion.draw('mpl'))

## Paso 2 — Alice y Bob crean su recurso compartido

Alice aplica H a su qubit y después CNOT hacia el qubit de Bob.

Esto crea Φ⁺ entre los registros `Alice` y `Bob`. El qubit `Mensaje` todavía no participa.

In [ ]:
# Crear el estado de Bell entre Alice y Bob.
teleportacion.h(qubit_alice[0])
teleportacion.cx(qubit_alice[0], qubit_bob[0])
teleportacion.barrier()

display(teleportacion.draw('mpl'))

## Paso 3 — Alice combina el mensaje con su qubit

Alice realiza dos operaciones locales:

1. CNOT desde `Mensaje` hacia `Alice`.
2. H sobre `Mensaje`.

Bob todavía no realiza ninguna operación.

In [ ]:
# Alice combina sus dos qubits.
teleportacion.cx(mensaje[0], qubit_alice[0])
teleportacion.h(mensaje[0])
teleportacion.barrier()

display(teleportacion.draw('mpl'))

## Paso 4 — Alice mide sus dos qubits

Alice obtiene dos resultados clásicos:

| Medición | Se guarda en |
|---|---|
| Qubit `Mensaje` | `bits_Alice[0]` |
| Qubit `Alice` | `bits_Alice[1]` |

Después de medir, el estado original ya no permanece en el qubit `Mensaje`.

In [ ]:
# Alice mide el mensaje y su mitad del par.
teleportacion.measure(mensaje[0], bits_alice[0])
teleportacion.measure(qubit_alice[0], bits_alice[1])
teleportacion.barrier()

display(teleportacion.draw('mpl'))

## Paso 5 — Alice envía dos bits clásicos

Alice puede obtener `00`, `01`, `10` o `11`. Envía esos dos bits a Bob por un canal clásico.

| Resultado de Alice | Acción de Bob |
|:--:|---|
| `00` | No aplica ninguna compuerta |
| `01` | Aplica Z |
| `10` | Aplica X |
| `11` | Aplica X y Z |

> Estos son bits normales. Bob debe recibirlos antes de poder recuperar el estado.

## Paso 6 — Bob aplica las correcciones

En Qiskit, `if_test` significa: “aplica esta compuerta solamente si el bit recibido vale 1”.

In [ ]:
# Si la medición del qubit Alice fue 1, Bob aplica X.
with teleportacion.if_test((bits_alice[1], 1)):
    teleportacion.x(qubit_bob[0])

# Si la medición del Mensaje fue 1, Bob aplica Z.
with teleportacion.if_test((bits_alice[0], 1)):
    teleportacion.z(qubit_bob[0])

teleportacion.barrier()
display(teleportacion.draw('mpl'))

## Comprobación — Medir el qubit de Bob

Finalmente medimos el qubit `Bob`. Si el protocolo funcionó, Bob debe obtener siempre `1`.

In [ ]:
# Guardar la medición final de Bob.
teleportacion.measure(qubit_bob[0], bit_bob[0])

display(teleportacion.draw('mpl'))

In [ ]:
# Ejecutar el protocolo completo.
teleportacion_preparada = transpile(teleportacion, simulador)
conteos_teleportacion = simulador.run(
    teleportacion_preparada,
    shots=1000,
    seed_simulator=7
).result().get_counts()

print(conteos_teleportacion)
display(plot_histogram(
    conteos_teleportacion,
    title="Teleportación del estado |1⟩"
))

## ¿Cómo leemos los resultados?

Qiskit los muestra con el formato:

```text
bit_Bob bits_Alice
```

Por ejemplo:

```text
1 10
```

significa que:

- Bob obtuvo `1`;
- Alice obtuvo `10`.

Los bits de Alice cambian, pero todos los resultados comienzan con `1`. Esto confirma que Bob recuperó el estado enviado.

## 🧠 Lo más importante

- Alice no conoce necesariamente el estado que envía.
- La medición elimina la copia original.
- Bob no recibe el qubit físico de Alice.
- Bob necesita el par entrelazado y los dos bits clásicos.
- Por eso la teleportación no envía información más rápido que la luz.

---

# 🧪 Ejercicios

Completa solamente las líneas `TODO`. Las respuestas están en el solucionario docente.

## Ejercicio 1 — Construir Φ⁻

El circuito ya crea Φ⁺. Agrega la compuerta necesaria para cambiar la fase.

In [ ]:
ejercicio_1 = QuantumCircuit(2)
ejercicio_1.h(0)
ejercicio_1.cx(0, 1)

# TODO: agrega Z para obtener Φ⁻.
# ejercicio_1.____(___)

estado_e1 = Statevector.from_instruction(ejercicio_1)
display(ejercicio_1.draw('mpl'))
display(estado_e1.draw('latex'))

## Ejercicio 2 — Construir Ψ⁺

El circuito ya crea Φ⁺. Agrega la compuerta necesaria para que los resultados sean opuestos.

In [ ]:
ejercicio_2 = QuantumCircuit(2)
ejercicio_2.h(0)
ejercicio_2.cx(0, 1)

# TODO: agrega X para obtener Ψ⁺.
# ejercicio_2.____(___)

estado_e2 = Statevector.from_instruction(ejercicio_2)
display(ejercicio_2.draw('mpl'))
display(estado_e2.draw('latex'))

## Ejercicio 3 — Identificar a Alice y Bob

Completa con tus palabras:

- El qubit `Mensaje` sirve para:
- El qubit `Alice` sirve para:
- El qubit `Bob` sirve para:
- Los dos bits clásicos sirven para:

## Ejercicio 4 — Teleportar `|0⟩`

Vuelve a ejecutar el protocolo sin aplicar X al qubit `Mensaje`.

1. ¿Con qué estado comienza el mensaje?
2. ¿Qué valor debe obtener Bob?
3. ¿Los resultados de Alice siguen cambiando?

---

# ✅ Cierre del módulo

En este notebook aprendimos a:

- construir Φ⁺, Φ⁻, Ψ⁺ y Ψ⁻;
- distinguir correlación y fase;
- identificar los qubits de Alice y Bob;
- separar la teleportación en preparación, entrelazamiento, medición, comunicación y corrección;
- comprobar que Bob recupera el estado.

## 📚 Referencias

- [IBM Quantum Learning — Teleportation](https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information/entanglement-in-action/qiskit-implementation)
- [IBM Quantum — Control clásico](https://quantum.cloud.ibm.com/docs/en/guides/classical-feedforward-and-control-flow)

---